# 04 – Matching resumes with job offers

For a resume, rank the 853 job descriptions by similarity and return the top 5.

Two representations are compared: **TF-IDF cosine** and **sentence-embedding cosine**.

**Evaluation.** There is no ground truth (no resume is labelled with the offers it fits). Proxy used:
each job description gets a category from the resume classifier of notebook 03 (trained on resumes only);
a recommended offer counts as *relevant* when its category equals the true category of the resume.
**Precision@5** = share of relevant offers among the 5 recommended. A manual check of a few examples completes it.

In [1]:
import sys
sys.path.insert(0, "..")          # the shared code lives in ../matcher
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from matcher.data import load_resumes, load_jobs
from matcher.models import split, tfidf_classifier, embed_documents

pd.set_option("display.max_colwidth", 120)

In [2]:
resumes, jobs = load_resumes(), load_jobs()
X_train, X_test, y_train, y_test = split(resumes)
y_true = y_test.to_numpy(dtype=object)     # plain numpy array of labels
print(len(jobs), "job descriptions")
jobs.head(3)

842 job descriptions


,title,company,description,text
0,Sales Specialist,Google,minimum qualifications\nbachelors degree or equivalent practical experience years of experience in saas or productiv...,sales specialist minimum qualifications bachelors degree or equivalent practical experience years of experience in s...
1,Apple Solutions Consultant,Apple,description\nas an asc you will be highly influential in growing mind and market share of apple products while build...,apple solutions consultant description as an asc you will be highly influential in growing mind and market share of ...
2,Licensing Coordinator - Consumer Products,Netflix,its an amazing time to be joining netflix as we continue to transform entertainment globally netflix is the worlds l...,licensing coordinator consumer products its an amazing time to be joining netflix as we continue to transform entert...


## 1. Give each job description a category (for the evaluation only)

In [3]:
clf = tfidf_classifier().fit(X_train, y_train)
jobs["category"] = clf.predict(jobs["text"])
coverage = jobs["category"].value_counts()
print(f"{coverage.size} of the 43 categories have at least one job offer")
coverage.head(15)

39 of the 43 categories have at least one job offer


category
Sales                        127
Management                   124
Digital Media                 80
Advocate                      76
Human Resources               44
Education                     42
Banking                       32
Consultant                    25
Apparel                       25
BPO                           25
Finance                       23
Web Designing                 21
Building and Construction     21
Testing                       20
Accountant                    17
Name: count, dtype: int64

In [4]:
# Is the category of a job sensible? Look at titles per category
jobs.groupby("category")["title"].agg(lambda t: " | ".join(t.head(4))).head(15)

category
Accountant                   Office Clerk | General Office Clerk | Holman Frenia Allison, PC Hiring for Audit Staff Accountant Job at... | Accoun...
Advocate                                              Registered Nurse | Registered Nurse (RN-CA02-007) | Registered Nurse | Criminal Investigator 2
Apparel                                Retail Sales Associate | Retail Sales Associate | Sales Associate - Easton T/C | Sales Associate - Sun Center
Architecture                 Senior Software Engineer | Staff Software Engineer - SMB - Marketing Enablement | Software Engineer (Remote) | Senio...
Arts                                                            Art Teacher | Advertising Design Intern - 2022 | Art Director | Animation Programmer
Automobile                   Kitchen Cleaner | Kitchen Exhaust Associate (Commerce, CA) | Business Development Center Representative - Performanc...
Aviation                                                                  Overnight Janitor - Los

## 2. Two matching methods

In [5]:
# TF-IDF fitted on resumes + jobs, so that both are in the same vector space
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True, stop_words="english")
vec.fit(pd.concat([X_train, jobs["text"]]))
T_jobs, T_test = vec.transform(jobs["text"]), vec.transform(X_test)

E_jobs = embed_documents(jobs["text"], name="jobs_chunks")
E_test = embed_documents(X_test, name="test_chunks")

def top_k(scores, k=5):
    return np.argsort(-scores, axis=1)[:, :k]

S_tfidf = (T_test @ T_jobs.T).toarray()
S_emb = E_test @ E_jobs.T
# Hybrid: both similarities rescaled by their best value for the resume, then averaged
S_hybrid = S_tfidf / S_tfidf.max(axis=1, keepdims=True) + S_emb / S_emb.max(axis=1, keepdims=True)

tops = {"TF-IDF cosine": top_k(S_tfidf), "Embedding cosine": top_k(S_emb), "Hybrid": top_k(S_hybrid)}

## 3. Precision@5

The categories of the jobs come from a classifier, and that classifier could favour the matching method that uses the same features
(TF-IDF labels may favour TF-IDF matching). To check this bias, the jobs are labelled twice: by the TF-IDF classifier and by the embedding classifier.

In [6]:
from matcher.models import embedding_classifier

E_train = embed_documents(X_train, name="train_chunks")
labels = {
    "jobs labelled by TF-IDF classifier": jobs["category"].to_numpy(dtype=object),
    "jobs labelled by embedding classifier": embedding_classifier().fit(E_train, y_train).predict(E_jobs).astype(object),
}
covered = y_test.isin(coverage[coverage >= 5].index).values      # resumes whose category has at least 5 offers

def precision_at_5(top, job_cat):
    return (job_cat[top] == y_true[:, None]).mean(axis=1)

table = pd.DataFrame({lab: {m: precision_at_5(t, cats)[covered].mean() for m, t in tops.items()} for lab, cats in labels.items()})
table.loc["random"] = [np.mean([(cats == c).mean() for c in y_true[covered]]) for cats in labels.values()]
print(f"Precision@5 on the {covered.sum()} test resumes whose category has at least 5 offers")
table.round(3)

Precision@5 on the 1841 test resumes whose category has at least 5 offers


,jobs labelled by TF-IDF classifier,jobs labelled by embedding classifier
TF-IDF cosine,0.362,0.308
Embedding cosine,0.283,0.284
Hybrid,0.369,0.330
random,0.033,0.032


In [7]:
# Precision@5 per category (jobs labelled by the TF-IDF classifier)
cats = labels["jobs labelled by TF-IDF classifier"]
per_cat = pd.DataFrame({"category": y_true, **{m: precision_at_5(t, cats) for m, t in tops.items()}})[covered]
per_cat.groupby("category").mean().round(2).sort_values("Hybrid", ascending=False)

,TF-IDF cosine,Embedding cosine,Hybrid
category,,,
Human Resources,0.90,0.52,0.89
Education,0.81,0.85,0.86
Accountant,0.74,0.50,0.74
Digital Media,0.74,0.59,0.74
Sales,0.63,0.56,0.64
Building and Construction,0.56,0.58,0.63
DevOps,0.52,0.66,0.60
Network Security Engineer,0.49,0.44,0.53
Advocate,0.51,0.45,0.51


## 4. Manual check

In [8]:
examples = [np.where(y_true == c)[0][0] for c in ["Data Science", "Human Resources", "Sales"]]
for i in examples:
    print(f"=== {y_true[i]}: {X_test.iloc[i][:150]}...")
    print(pd.DataFrame({m: jobs["title"].values[t[i]] for m, t in tops.items()}).to_string(), "\n")

=== Data Science: links professional summary jessica claire 100 montgomery st 10th floor 555 4321000 resumesampleexamplecom innovative meticulous professional expertise...
                                                 TF-IDF cosine                          Embedding cosine                                       Hybrid
0                                          Senior Data Analyst                       Senior Data Analyst                          Senior Data Analyst
1                        Engineering Lead - Analytics Platform            Data Analytics Senior Mangager        Engineering Lead - Analytics Platform
2  Senior Software Engineer -  Virtualization Group - (Remote)   Vice President of Business Intelligence  Senior Full Stack Software Engineer- Remote
3                                       Card Fraud/Servcg Spec  Business Analyst, ACS Publications Sales                         Supply Chain Analyst
4                                  Marketing Analytics Manager                

In [9]:
# Similarity scores: how confident is the match?
best = S_emb.max(axis=1)
pd.Series(best).describe().round(3)

count    2417.000
mean        0.677
std         0.062
min         0.311
25%         0.643
50%         0.687
75%         0.720
max         0.820
dtype: float64

## Findings

Precision@5 on the 1,841 test resumes whose category has at least 5 offers:

| Method | Jobs labelled by TF-IDF classifier | Jobs labelled by embedding classifier |
|--------|-----|-----|
| TF-IDF cosine | 0.362 | 0.308 |
| Embedding cosine | 0.283 | 0.284 |
| **Hybrid** | **0.369** | **0.330** |
| Random | 0.033 | 0.032 |

- All methods are about 10 times better than random. The **hybrid** (sum of both similarities, each rescaled by its maximum) is the best under both labellings.
- The bias check matters: with TF-IDF labels, TF-IDF matching was favoured (+0.08 over embeddings). With embedding labels the gap shrinks to +0.02, but TF-IDF stays ahead, so the conclusion does not depend on the labelling.
- Embeddings are better for some categories (Education, DevOps, Database), TF-IDF for others (Human Resources 0.90 vs 0.52, Accountant): combining them takes the best of both.
- **Limits of the data:** the 853 offers do not cover every field (4 of the 43 categories have no offer at all), and the labels given to the jobs are noisy (nurses labelled "Advocate", a janitor labelled "Aviation"). For a Data Science resume, only a few offers are data jobs. The precision is therefore a lower bound, and the app warns the user when the best similarity is low.
- The manual check is convincing for well-covered fields: a Human Resources resume gets 5 HR offers (HR director, recruiters, HR generalist).
- **Choice for the app: hybrid matching.**